# Поступашки: маркетинговый анализ, attribution и ROMI

Этот ноутбук — продолжение отдельного EDA. Здесь нет повторного исследования данных: только подготовка необходимых агрегатов, сопоставление продаж с маркетинговыми кампаниями, оценка uplift, attribution и ROMI.

**Файлы для запуска:** `base.xlsx` и `marketing_posts.csv`.

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from scipy import stats

# Минимальная подготовка данных из base.xlsx — без повторения EDA
df = pd.read_excel('base.xlsx')
df.columns = ['student_id', 'amount', 'course', 'ts']
df['date'] = df['ts'].dt.date

orders = df.groupby(['student_id', 'ts']).agg(
    amount=('amount', 'sum'),
    n_items=('course', 'count'),
    date=('date', 'first'),
).reset_index()

daily = orders.groupby('date').agg(
    orders=('student_id', 'count'),
    revenue=('amount', 'sum'),
    items=('n_items', 'sum'),
).reset_index()

# Последний день исходного файла неполный, поэтому исключаем его из сравнений.
daily_full = daily[daily['date'] != daily['date'].max()].copy()


---

## 1. Сопоставление продаж с маркетинговой историей

Ниже проверяем на маркетинговой истории гипотезу из отдельного EDA о связи всплесков продаж с лончами и промо на реальных постах, которые
удалось восстановить вручную из публичного Telegram (`t.me/s/postypashki_old`)
и TGStat (`marketing_posts.csv`, лежит рядом с этим ноутбуком).

**Важная оговорка про метод**, чтобы не выдать желаемое за
измеренное: список постов в этом файле — не полная лента канала
день за днём, а **целенаправленно найденные** посты вокруг уже
известных из отдельного EDA всплесков продаж. Поэтому сравнение
"есть пост / нет поста" ниже — это не независимая проверка гипотезы
"посты вызывают продажи", а **количественное подтверждение размера**
уже найденной связи. Причинность отсюда не следует — только то, что
величина связи не в пределах шума.

### 1.1 Загрузка кампаний

In [14]:
posts = pd.read_csv('marketing_posts.csv')
posts['date'] = pd.to_datetime(posts['date']).dt.date
posts_campaign = posts[posts['campaign_id'] != 'none'].copy()

print(posts.shape, "постов всего,", len(posts_campaign), "относятся к кампаниям")
posts_campaign[['campaign_id', 'date', 'type', 'views', 'clicks_fwd']]

(17, 14) постов всего, 15 относятся к кампаниям


           campaign_id        date              type    views  clicks_fwd
0           start_sale  2026-08-08       sale_launch  26900.0        98.0
3           start_sale  2026-08-10     sale_extended  16400.0         8.0
4           pro_launch  2026-08-22            launch  35400.0       152.0
5       tbank_deadline  2026-08-31   content_trigger  20500.0       316.0
6       tbank_deadline  2026-09-01   engagement_hook  17200.0       364.0
7       tbank_deadline  2026-09-02   content_trigger  16900.0       272.0
8       tbank_deadline  2026-09-03           urgency  14900.0       298.0
9       tbank_deadline  2026-09-05           unknown  15300.0        50.0
10      tbank_deadline  2026-09-06           urgency  12500.0       246.0
11  partner_crosspromo  2026-09-06       lead_magnet  14400.0        94.0
12      tbank_deadline  2026-09-06           content  11800.0       239.0
13      tbank_deadline  2026-09-06      sale_content  10900.0       442.0
14      tbank_deadline  2026-09-06    

### 1.2 Выручка в дни с кампанией vs без неё

In [15]:
campaign_days = set(posts_campaign['date'])
daily_full['has_campaign_post'] = daily_full['date'].isin(campaign_days)

with_post = daily_full.loc[daily_full.has_campaign_post, 'revenue']
without_post = daily_full.loc[~daily_full.has_campaign_post, 'revenue']

t_stat2, p_val2 = stats.ttest_ind(with_post, without_post, equal_var=False)
df_welch2 = (with_post.var() / len(with_post) + without_post.var() / len(without_post)) ** 2 / (
    (with_post.var() / len(with_post)) ** 2 / (len(with_post) - 1)
    + (without_post.var() / len(without_post)) ** 2 / (len(without_post) - 1)
)
t_crit2 = stats.t.ppf(0.975, df_welch2)

print(f"Дни с постом кампании:  n={len(with_post)}, "
      f"mean revenue={with_post.mean():.0f} ₽, sd={with_post.std():.0f}")
print(f"Дни без поста:          n={len(without_post)}, "
      f"mean revenue={without_post.mean():.0f} ₽, sd={without_post.std():.0f}")
print(f"Welch t-test: t={t_stat2:.2f}, df={df_welch2:.1f}, "
      f"t_crit(α=0.05, двусторонний)={t_crit2:.2f}, p={p_val2:.5f}")

Дни с постом кампании:  n=10, mean revenue=261664 ₽, sd=101047
Дни без поста:          n=27, mean revenue=121411 ₽, sd=114075
Welch t-test: t=3.62, df=18.1, t_crit(α=0.05, двусторонний)=2.10, p=0.00195


**Находка:** средняя дневная выручка в дни с найденным постом кампании
выше, чем в остальные дни, и разница статистически значима
(|t| > t_crit). Это ожидаемо — именно вокруг этих дат мы и искали
посты, — но теперь у нас есть число, а не "на глаз похоже".

**Решение:** использовать этот результат как **подтверждение**
гипотезы из отдельного EDA, а не как её независимое доказательство. В
презентации формулировать аккуратно: "выручка в дни известных кампаний
статистически выше базовой" — не "реклама вызывает продажи".

### 1.3 Охват кампании vs прирост выручки в её окне

**Проблема:** охват (просмотры/переходы) и выручка измеряются в разных
единицах и по разным датам — просмотры у поста, выручка у заказа.
Нужен способ их сопоставить, не выдумывая atrribution на уровне
пользователя (мы уже установили в предыдущем разделе, что это
невозможно на этих данных).

**Решение:** сравнивать не пользователей, а **окна**: для каждой из 3
известных кампаний берём (а) суммарный охват её постов (views + клики)
и (б) фактическую выручку в дни всплеска из отдельного EDA минус базовая
(медианная) выручка обычного дня — это и есть грубая оценка "прироста"
за счёт кампании, при условии что в этом окне не было других
конкурирующих объяснений.

In [16]:
baseline_days = daily_full.loc[~daily_full.has_campaign_post, 'revenue']
baseline_daily_revenue = baseline_days.median()

spike_windows = {
    'start_sale': (pd.to_datetime('2026-08-08').date(), pd.to_datetime('2026-08-10').date()),
    'pro_launch': (pd.to_datetime('2026-08-22').date(), pd.to_datetime('2026-08-24').date()),
    'tbank_deadline': (pd.to_datetime('2026-09-04').date(), pd.to_datetime('2026-09-06').date()),
}

rows = []
for camp, (start, end) in spike_windows.items():
    window_revenue = daily_full.loc[
        (daily_full['date'] >= start) & (daily_full['date'] <= end), 'revenue'
    ].sum()
    n_days = (end - start).days + 1
    uplift = window_revenue - baseline_daily_revenue * n_days

    # для tbank_deadline считаем охват вместе с partner_crosspromo —
    # это один и тот же информационный повод (пост 6 сен), а не
    # отдельная кампания
    camp_ids = [camp] if camp != 'tbank_deadline' else ['tbank_deadline', 'partner_crosspromo']
    reach = posts_campaign.loc[
        posts_campaign['campaign_id'].isin(camp_ids), ['views', 'clicks_fwd']
    ].sum().sum()

    rows.append({
        'campaign': camp,
        'window': f"{start}..{end}",
        'reach_views_clicks': int(reach),
        'window_revenue': round(window_revenue),
        'baseline_revenue_for_window': round(baseline_daily_revenue * n_days),
        'uplift_revenue': round(uplift),
        'uplift_per_1000_reach': round(uplift / (reach / 1000), 1) if reach else None,
    })

campaign_summary = pd.DataFrame(rows)
campaign_summary

         campaign                  window  reach_views_clicks  window_revenue  baseline_revenue_for_window  uplift_revenue  uplift_per_1000_reach
0      start_sale  2026-08-08..2026-08-10               43406         1004697                       230070          774627                17846.1
1      pro_launch  2026-08-22..2026-08-24               35552          829718                       230070          599648                16866.8
2  tbank_deadline  2026-09-04..2026-09-06              166471         1150344                       230070          920274                 5528.1

**Находка:** во всех трёх окнах фактическая выручка заметно выше
базовой (положительный `uplift_revenue`), а `uplift_per_1000_reach`
даёт сопоставимую по кампаниям "эффективность охвата" — сколько рублей
прироста приходится на 1000 просмотров/переходов. Это позволяет
сравнивать кампании между собой даже без единого рубля данных о
стоимости размещения.

**Решение и явная граница метода:** `uplift_per_1000_reach` — это
**не ROMI**. Явно проговариваем разницу дальше.

### 1.4 Как считать ROMI — что можно сейчас и чего не хватает

**Целевая формула (задана в кейсе):**

```
ROMI = (Attributed Revenue − Marketing Cost) / Marketing Cost × 100%
```

**Проблема:** у нас нет ни одного из двух слагаемых в чистом виде.
- `Attributed Revenue` требует attribution на уровне пользователя
  (клик/касание → лид → оплата), а мы установили в разделе про
  атрибуцию (после Задачи 2), что связать `student_id` с конкретным
  постом/каналом невозможно — нет tracking-ключа и нет права сшивать
  анонимный id с реальным Telegram-аккаунтом.
- `Marketing Cost` отсутствует полностью: все найденные кампании — это
  посты в собственных каналах (не платное размещение), у них нет
  медиа-бюджета, а издержки на производство контента исторически не
  логировались. Найти реальное **платное** внешнее размещение с
  известной ценой за период кейса не удалось (см. раздел про сторонние
  каналы) — соответственно, для 100% найденных кампаний знаменатель
  формулы буквально неоткуда взять, а не "посчитан неточно".

**Решение:**
1. Сейчас вместо ROMI считаем прокси — `uplift_per_1000_reach` из 6.3.
   Это направленный (не денежный) показатель эффективности, годится
   для сравнения кампаний между собой ("охват вокруг ПРО-лонча
   конвертировался в выручку лучше/хуже, чем охват вокруг СТАРТа"), но
   **не годится** как ответ на "куда направить следующий бюджет в
   рублях" — это и есть главный бизнес-вопрос кейса (слайд 8), и
   честный ответ на него сегодня: "нельзя посчитать, вот почему".
2. Чтобы формула заработала в будущем (зона Задачи 5), в модель данных
   должны попасть **до** запуска кампании, а не восстанавливаться
   постфактум:
   - `cost` и `publication_time` на уровне каждого `placement` — в том
     числе для постов в собственных каналах (учётная, а не рыночная
     стоимость: например, часы автора/дизайнера);
   - уникальный `campaign_id`/`placement_id`/`creative_id` в каждой
     ссылке, которую пользователь видит в посте (это уже прямо
     сформулировано в Задаче 5 кейса);
   - join-ключ от клика по этой ссылке до `payment`, то есть решённый
     user stitching (Задача 4) — без него `Attributed Revenue`
     физически не из чего собрать.

Итог для передачи в Задачи 3–5: ROMI по историческим данным — это
**нерешаемая задача**, а не "мы не успели посчитать". Всё, что можно
сделать без выдумывания — это `uplift_per_1000_reach`, использовать
дальше только его, и не подставлять эту цифру туда, где кейс просит
именно ROMI.

---

## 2. MVP: рабочий прототип attribution + ROMI

Раздел 1 показал, что ROMI **на реальных цифрах прошлого** посчитать
нельзя — ни числителя (Attributed/Incremental Revenue на уровне
пользователя), ни знаменателя (Marketing Cost) взять неоткуда. Но
кейс (слайд 20, "MVP обязателен") просит не готовую цифру, а
**рабочую систему**: код, который решает реальный кусок цепочки
измерения и явно помечает, где данные настоящие, а где — синтетика
для демонстрации логики. Это разные требования, и ниже — именно
второе.

**Что в этом разделе реально, а что синтетика:**
- `uplift_revenue` (числитель ROMI_inc) — **реальные данные**, взято
  из раздела 1.3 без изменений.
- Touch-уровневые данные (кто, по какой ссылке, когда кликнул) —
  **синтетика**: таких данных исторически нет вообще, они нужны только
  чтобы показать, как работает сама attribution-модель.
- Marketing Cost для ROMI_attr — гибрид: число постов кампании
  реальное (`marketing_posts.csv`), а часы на один пост и внутренняя
  ставка автора — SYNTHETIC-допущение (это никогда не логировалось).
  Для ROMI_inc (реальный `uplift_revenue`) cost вообще не передаём —
  функция честно возвращает `None`, а не 0 и не выдумку.

### 2.1 Attribution model

**Решение по модели:** беру **last-touch в окне 7 дней**. Обоснование:
у нас нет ни одного реального примера мультитач-пути (одновременных
разных касаний одного человека), поэтому любая модель сложнее
last-touch (linear, time-decay, position-based) распределяла бы
выручку по весам, которые мы бы просто придумали — а слайд 15 прямо
предупреждает: сложная модель не значит лучшая. Окно 7 дней — по
наблюдаемому в разделе 6 лагу "пост кампании → рост выручки" (1-3 дня
до пика, беру с запасом).

In [17]:
def attribution_model(touches, purchase_ts, purchase_revenue, model='last_touch', window_days=7):
    """
    touches: список (channel, touch_ts) для одного покупателя.
    Возвращает {channel: attributed_revenue} по выбранной модели,
    учитывая только касания в пределах window_days до покупки.
    """
    window_start = purchase_ts - pd.Timedelta(days=window_days)
    eligible = [(ch, ts) for ch, ts in touches if window_start <= ts <= purchase_ts]
    if not eligible:
        return {'organic_or_out_of_window': purchase_revenue}

    eligible.sort(key=lambda x: x[1])
    result = {}
    if model == 'last_touch':
        result[eligible[-1][0]] = purchase_revenue
    elif model == 'first_touch':
        result[eligible[0][0]] = purchase_revenue
    elif model == 'linear':
        share = purchase_revenue / len(eligible)
        for ch, _ in eligible:
            result[ch] = result.get(ch, 0) + share
    else:
        raise ValueError(f"неизвестная модель: {model}")
    return result

### 2.2 Синтетический пример на нескольких моделях

Ниже — придуманные (SYNTHETIC) покупатели и их касания, только чтобы
показать: выбор модели реально меняет, какому каналу "достаётся"
выручка. Каналы взяты из реальных `campaign_id` раздела 1, суммы и
пути — синтетика.

In [18]:
np.random.seed(42)  # SYNTHETIC ниже — фиксирую сид для воспроизводимости примера

synthetic_customers = [
    {  # мультитач: увидел лонч ПРО, потом дедлайн Т-Банка, купил
        'revenue': 14795,
        'purchase_ts': pd.Timestamp('2026-09-06 20:00'),
        'touches': [('pro_launch', pd.Timestamp('2026-08-22 13:19')),
                    ('tbank_deadline', pd.Timestamp('2026-09-03 19:06'))],
    },
    {  # одно касание — старт-распродажа
        'revenue': 6490,
        'purchase_ts': pd.Timestamp('2026-08-09 10:00'),
        'touches': [('start_sale', pd.Timestamp('2026-08-08 12:13'))],
    },
    {  # три касания подряд перед покупкой в дедлайн
        'revenue': 8950,
        'purchase_ts': pd.Timestamp('2026-09-06 21:00'),
        'touches': [('tbank_deadline', pd.Timestamp('2026-09-01 20:40')),
                    ('tbank_deadline', pd.Timestamp('2026-09-03 19:06')),
                    ('partner_crosspromo', pd.Timestamp('2026-09-06 15:06'))],
    },
    {  # покупка вне 7-дневного окна любого касания -> органика
        'revenue': 9990,
        'purchase_ts': pd.Timestamp('2026-08-30 12:00'),
        'touches': [('pro_launch', pd.Timestamp('2026-08-22 13:19'))],
    },
]

comparison = {}
for model in ['first_touch', 'last_touch', 'linear']:
    totals = Counter()
    for cust in synthetic_customers:
        attributed = attribution_model(
            cust['touches'], cust['purchase_ts'], cust['revenue'], model=model
        )
        for ch, rev in attributed.items():
            totals[ch] += rev
    comparison[model] = dict(totals)

pd.DataFrame(comparison).fillna(0)

                          first_touch  last_touch        linear
tbank_deadline                23745.0       14795  20761.666667
start_sale                     6490.0        6490   6490.000000
organic_or_out_of_window       9990.0        9990   9990.000000
partner_crosspromo                0.0        8950   2983.333333

**Находка (на синтетике):** first-touch отдаёт всю выручку 3-го
покупателя каналу `tbank_deadline` (первое касание), last-touch —
каналу `partner_crosspromo` (последнее касание перед покупкой), linear
делит между обоими. Разница по каналу может быть в разы — это
буквально то, о чём предупреждает слайд 15 ("сравните подходы").

**Решение:** в проде фиксируем last-touch/7 дней как стартовую модель
(см. 2.1), но код держим модель-агностичным (`model=...` параметр) —
чтобы поменять на linear/position-based одной строкой, когда появятся
реальные touch-данные.

### 2.3 ROMI_attr и ROMI_inc — рабочие функции

**Метод расчёта cost (для ROMI_attr ниже):** размещения — это посты в
собственных каналах, а не платная реклама, поэтому cost — это
альтернативные издержки на создание поста (см. п. 1.4, вариант 2):
`cost = число_постов × часы_на_пост × ставка_часа`. Число постов
кампании берём **реальное** (`marketing_posts.csv`), а часы на один
пост и внутреннюю ставку автора — как **SYNTHETIC**-допущение, потому
что это никогда не логировалось.

In [19]:
def compute_romi(attributed_or_incremental_revenue, marketing_cost):
    """
    ROMI = (Revenue - Cost) / Cost.
    Если cost неизвестен — возвращаем None, а не 0 и не выдумку.
    """
    if marketing_cost is None or marketing_cost == 0:
        return None
    return (attributed_or_incremental_revenue - marketing_cost) / marketing_cost

# ROMI_attr — на синтетических touch-данных 7.2. Cost = n_posts (РЕАЛЬНЫЕ,
# из marketing_posts.csv) × часы_на_пост × ставка_часа (SYNTHETIC-допущения)
n_posts_by_channel = posts_campaign['campaign_id'].value_counts().to_dict()

HOURS_PER_POST = 1.5  # SYNTHETIC: часы на текст + картинку + согласование
HOURLY_RATE = 800      # SYNTHETIC: внутренняя ставка автора/smm, ₽/час

synthetic_cost_by_channel = {
    ch: round(n * HOURS_PER_POST * HOURLY_RATE)
    for ch, n in n_posts_by_channel.items()
}
print("Cost по каналам (n_posts — реальные, часы и ставка — SYNTHETIC):")
print(synthetic_cost_by_channel)

romi_attr_example = {
    ch: compute_romi(rev, synthetic_cost_by_channel.get(ch))
    for ch, rev in comparison['last_touch'].items() if ch != 'organic_or_out_of_window'
}
print()
print("ROMI_attr (revenue — на СИНТЕТИЧЕСКИХ touch-данных 7.2, cost — по методу "
      "'часы×ставка' выше, только для проверки логики):")
print(romi_attr_example)

# ROMI_inc, вариант A — на РЕАЛЬНОМ uplift_revenue из 6.3, cost неизвестен:
# функция должна честно отказаться считать, а не подставить 0.
print()
print("ROMI_inc, если cost НЕ известен (revenue — РЕАЛЬНЫЕ данные из 6.3):")
for _, row in campaign_summary.iterrows():
    romi_inc = compute_romi(row['uplift_revenue'], marketing_cost=None)
    print(f"  {row['campaign']}: incremental_revenue={row['uplift_revenue']} ₽, "
          f"cost=неизвестен -> ROMI_inc={romi_inc}")

# ROMI_inc, вариант B — та же РЕАЛЬНАЯ revenue, но cost оценён тем же
# гибридным методом, что и для ROMI_attr выше (n_posts РЕАЛЬНЫЕ,
# часы/ставка SYNTHETIC-допущение). Это не измеренный ROMI, а оценка
# при явно названном допущении — но именно она пригодна для сравнения
# кампаний между собой при распределении будущего бюджета.
print()
print("ROMI_inc, если cost ОЦЕНЁН по методу 'часы×ставка' (revenue реальная, cost synthetic):")
for _, row in campaign_summary.iterrows():
    camp_ids = [row['campaign']] if row['campaign'] != 'tbank_deadline' \
        else ['tbank_deadline', 'partner_crosspromo']
    cost_estimate = sum(synthetic_cost_by_channel.get(c, 0) for c in camp_ids)
    romi_inc_est = compute_romi(row['uplift_revenue'], cost_estimate)
    print(f"  {row['campaign']}: incremental_revenue={row['uplift_revenue']} ₽ (реальные), "
          f"cost≈{cost_estimate} ₽ (synthetic) -> ROMI_inc≈{romi_inc_est:.2f}")

Cost по каналам (n_posts — реальные, часы и ставка — SYNTHETIC):
{'tbank_deadline': 12000, 'start_sale': 2400, 'pro_launch': 1200, 'partner_crosspromo': 1200, 'free_week': 1200}

ROMI_attr (revenue — на СИНТЕТИЧЕСКИХ touch-данных 7.2, cost — по методу 'часы×ставка' выше, только для проверки логики):
{'tbank_deadline': 0.23291666666666666, 'start_sale': 1.7041666666666666, 'partner_crosspromo': 6.458333333333333}

ROMI_inc, если cost НЕ известен (revenue — РЕАЛЬНЫЕ данные из 6.3):
  start_sale: incremental_revenue=774627 ₽, cost=неизвестен -> ROMI_inc=None
  pro_launch: incremental_revenue=599648 ₽, cost=неизвестен -> ROMI_inc=None
  tbank_deadline: incremental_revenue=920274 ₽, cost=неизвестен -> ROMI_inc=None

ROMI_inc, если cost ОЦЕНЁН по методу 'часы×ставка' (revenue реальная, cost synthetic):
  start_sale: incremental_revenue=774627 ₽ (реальные), cost≈2400 ₽ (synthetic) -> ROMI_inc≈321.76
  pro_launch: incremental_revenue=599648 ₽ (реальные), cost≈1200 ₽ (synthetic) -> ROMI_inc≈498

**Решение:** функция `compute_romi` показывает оба варианта на одной и
той же реальной `campaign_summary.uplift_revenue`:
- **cost не передан** → честно `None`, а не 0 (0 дал бы
  бесконечный/бессмысленный ROMI) и не выдуманное число;
- **cost передан как явно названное допущение** (гибридный метод
  "часы × ставка" из начала 2.3, где число постов — реальное) →
  функция без изменений в коде считает оценочный `ROMI_inc`, который
  уже можно использовать, чтобы сравнить кампании между собой при
  распределении 300 000 ₽ (главный бизнес-вопрос кейса) — не как
  единственно верную цифру, а как систему с явно проговоренными
  допущениями.

Как только бизнес начнёт реально логировать `cost` по Задаче 5 — та же
функция посчитает точный `ROMI_inc` без единой правки в коде.

### 2.4 Incrementality — что уже есть и что предложить дальше

**Attribution ≠ Incrementality** (слайд 16): attribution в 2.1–2.3
отвечает "кому мы приписали продажу", а не "случилась ли она вообще
благодаря рекламе". Наш `uplift_revenue` из 6.3 — это простейший
причинный дизайн из допустимого на слайде 16 списка: **interrupted
time series** (сравнение факта с базовой линией "обычного дня"). Он
реальный и уже посчитан — специально ставить дополнительный A/B-тест
на данные прошлого не нужно и невозможно: прошлое нельзя
рандомизировать задним числом.

**Что предложить на будущее** — и здесь как раз пригождается находка
про 5+ собственных каналов Поступашек (раздел про сеть каналов):
раз весь трафик — свой, можно поставить **randomized holdout** почти
бесплатно, без внешнего вендора:
- на следующий лонч случайно не показывать акцию ~15-20% аудитории
  одного из каналов (или использовать один канал сети как контроль, а
  структурно похожий — как treatment);
- сравнить конверсию/выручку в holdout-группе с остальными Welch
  t-test-ом — тем же методом, что уже применялся в разделе 4.1;
- это даст настоящий ROMI_inc с числителем, посчитанным не по
  interrupted time series (слабый дизайн), а по рандомизированному
  эксперименту (сильный дизайн) — то, что слайд 16 называет лучшим
  вариантом из списка.